First we do some quality checks with the data compared to the original intervals.

In [1]:
import numpy as np
from Historia.shared.design_utils import read_labels

data_path_new = f"/media/croderog/SeagateExpansionDrive/HCM/10GH00962/scenarios/6/MCMC_80_2timesUpper/data"
X_file_new_name = f"{data_path_new}/MCMC_samples_scenario_6_80_2timesUpper.dat"

data_path_old = f"/media/croderog/SeagateExpansionDrive/HCM/10GH00962/scenarios/6/data"
X_file_old_name = f"{data_path_old}/X.txt"

X_new = np.loadtxt(X_file_new_name,dtype=float)
X_old = np.loadtxt(X_file_old_name,dtype=float)

xlabels = read_labels(f"{data_path_old}/xlabels.txt")

min_X_new = np.round(np.min(X_new,axis=0),6)
min_X_old = np.round(np.min(X_old,axis=0),6)

max_X_new = np.round(np.max(X_new,axis=0),6)
max_X_old = np.round(np.max(X_old,axis=0),6)

perc_increase_min = np.round(100*(min_X_new-min_X_old)/min_X_old,2)
perc_increase_max = np.round(100*(max_X_new-max_X_old)/max_X_old,2)

for i, label in enumerate(xlabels):
	print(f"The interval for {label} is [{min_X_new[i]}, {max_X_new[i]}]. Compared to the original interval of [{min_X_old[i]}, {max_X_old[i]}] ([{perc_increase_min[i]}%, {perc_increase_max[i]}%])")

The interval for PCa_b is [0.000197, 0.000264]. Compared to the original interval of [2.1e-05, 0.000147] ([838.1%, 79.59%])
The interval for Tref is [164.02329, 284.877679]. Compared to the original interval of [60.0088, 224.985] ([173.33%, 26.62%])
The interval for perm50 is [0.108207, 0.471972]. Compared to the original interval of [0.175227, 0.524932] ([-38.25%, -10.09%])
The interval for CV_ventricles is [0.661694, 0.679904]. Compared to the original interval of [0.380048, 0.799992] ([74.11%, -15.01%])
The interval for a_ventricles is [0.809874, 4.899513]. Compared to the original interval of [1.00024, 4.99856] ([-19.03%, -1.98%])
The interval for EDP_lv is [3.391437, 9.574101]. Compared to the original interval of [1.00357, 7.49735] ([237.94%, 27.7%])
The interval for EDP_rv is [0.917538, 8.611945]. Compared to the original interval of [1.00466, 7.49384] ([-8.67%, 14.92%])
The interval for Rsys is [0.581524, 1.50427]. Compared to the original interval of [1.00252, 3.99864] ([-41.9

First we need to split the X file into the different fields.

In [2]:
from SIMULATION_library import simulator_utils

fields   = ["ToRORd","ToRORd_land","EP", "mechanics", "circadapt"]
idx_list = [[0],[1,2],[3],[4,5,6],[7,8]]
X_output_file_list = [f"{data_path_new}/X_{field}.txt" for field in fields]

simulator_utils.split_X(X_file = X_file_new_name,
						idx_list = idx_list,
						X_output_file_list = X_output_file_list)

Now we copy the adequate xlabels and create a new `X.txt`:

In [3]:
import os

X_array = []
xlabels_array = []

for field in fields:
	os.system(f"cp {data_path_old}/xlabels_{field}.txt {data_path_new}/.")
    
	X_ = np.loadtxt(f"{data_path_new}/X_{field}.txt", dtype=float)

    # Check if X_ has only one column, reshape to 2D array
	if X_.ndim == 1:
		X_ = X_.reshape(-1, 1)

	X_array.append(X_)

	xlabels_ = read_labels(f"{data_path_new}/xlabels_{field}.txt")
	xlabels_array.append(xlabels_)

# X = np.concatenate(X_array, axis=1)
    
X = np.hstack(X_array)
xlabels = np.concatenate(xlabels_array, axis=0)

np.savetxt(f"{data_path_new}/X.txt",X,fmt="%g")
np.savetxt(f"{data_path_new}/xlabels.txt",xlabels,fmt="%s")

We create the needed json files:

In [4]:
from SIMULATION_library import simulator_utils

os.makedirs(f"{data_path_new}/../json_files",exist_ok=True)
os.system(f"cp {data_path_old}/../json_files/default.json {data_path_new}/../json_files/.")
os.system(f"cp {data_path_old}/../json_files/tags.json {data_path_new}/../json_files/.")
os.system(f"cp {data_path_old}/../json_files/settings*.json {data_path_new}/../json_files/.")
os.system(f"cp {data_path_old}/../json_files/*clinic*.json {data_path_new}/../json_files/.")

simulator_utils.X_to_json(labels_fields = fields,
                          datafolder    = data_path_new,
                          outputfolder  = f"{data_path_new}/../json_files",
                          default_json  = f"{data_path_new}/../json_files/default.json")

mkdir: cannot create directory ‘/media/croderog/SeagateExpansionDrive/HCM/10GH00962/scenarios/6/MCMC_80_2timesUpper/data/../json_files’: File exists


generating json file...
ToRORd
ToRORd_land
EP
mechanics
circadapt
ToRORd
ToRORd_land
EP
mechanics
circadapt


100%|██████████| 50/50 [00:00<00:00, 591.36it/s]


Now you can run notebooks 1 and 2.